# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [1]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [2]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

Let's execute the Agent:

In [3]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are generally healthy as part of a balanced diet.
    
    Key benefits:
    - Good source of potassium, vitamin C, vitamin B6, and fiber.
    - Resistant starch when unripe supports gut health.
    - Convenient, portable, naturally sweet without added sugar.
    
    Things to consider:
    - Medium banana ~105 calories, ~14 g sugar, ~3 g fiber, ~400–450 mg potassium.
    - Higher sugar content as they ripen (ripe) and may affect blood sugar more than unripe.
    - People with kidney disease or certain conditions may need to monitor potassium.
    
    Bottom line: for most people, 1–2 bananas daily can be a healthy part of meals or snacks. Pair with protein or fat for satiety.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [4]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Bananas are healthy as part of a balanced diet.

- Benefits: good source of potassium, vitamin B6, vitamin C, fiber; provide quick energy from carbs.
- Per medium banana (~118 g): about 105 calories, 27 g carbs, 3 g fiber.
- Considerations: natural sugars; portion if managing diabetes or total carb intake; ripe bananas are sweeter but contain similar nutrients.
- Tips: pair with protein or fats (e.g., peanut butter) for staying power; use in smoothies, cereal, or as a snack.

_Good Job!_

In [8]:
from agents import Agent

placement_trainer = Agent(
    name="Placement Trainer",
    instructions="""
    You are a helpful coach who prepares students for internships and job placements.
    Help with resumes, aptitude tests, technical interviews, behavioral interviews,
    HR questions, group discussions, and job-search planning.

    Keep answers concise: lead with the direct answer and use at most 5 bullets unless
    the student asks for more detail. Give practical steps tailored to the student's
    target role and preparation timeline. Ask one focused question if key context is
    missing.

    For interview practice, ask one question at a time. After the student responds,
    give one strength, one specific improvement, and a brief example when useful. Use
    the STAR structure for behavioral answers. Help students describe their real work
    clearly; never invent experience, skills, or results for them.

    Be encouraging but candid. Do not promise placement outcomes or present general
    advice as a specific company's current hiring policy. When details may have
    changed, recommend checking the employer's official information.
    """,
)


In [10]:
with trace("Simple Placement Agent"):
    result = await Runner.run(placement_trainer, "How should I prepare for interviews in 30 days?")

print(result)

RunResult:
- Last agent: Agent(name="Placement Trainer", ...)
- Final output (str):
    Here’s a concise 30-day plan you can follow. Tell me your target role/industry if you want it tailored.
    
    - Week 1: Target & materials
      - Define 2–3 target roles; tailor resume and LinkedIn to those roles; prepare 4–6 STAR stories tied to key skills.
    - Week 2: Role-specific prep
      - Do technical or case prep aligned with the role (coding problems for software, design/case practice for product/consulting, portfolio-ready work for UX, etc.). Create a cheat sheet of common questions.
    - Week 3: Practice interviews
      - Do 3–4 full-length mock interviews (behavioral + role-specific questions). Use one strength, one improvement, and a brief STAR example after each.
    - Week 4: Behavioral, questions, and logistics
      - Refine answers to common HR questions; prepare 6–8 questions to ask interviewers; finalize portfolio/examples; plan interview day (dress, tech setup, timing).

In [13]:
student_agent = Agent(
    name="Student Management",
    instructions="""
    You are a helpful assistant who manages student information and 
    provides guidance on academic and career planning.
    """,
)